In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, zscore

In [ ]:
def normalize_file(input_file, output_file):
    df = pd.read_csv(input_file, sep="\t", header=None)

    df.columns = [
        "chr",
        "start",
        "end",
        "reads",
        "bases_covered",
        "length",
        "fraction"
    ]

    total_reads = df["reads"].sum()

    df["rpm"] = df["reads"] / total_reads * 1e6

    df.to_csv(output_file, sep="\t", index=False)


normalize_file("19.1mb.cov.txt", "19.1mb.rpm.txt")
normalize_file("20.1mb.cov.txt", "20.1mb.rpm.txt")
normalize_file("21.1mb.cov.txt", "21.1mb.rpm.txt")
normalize_file("23.1mb.cov.txt", "23.1mb.rpm.txt")
normalize_file("22.1mb.cov.txt", "22.1mb.rpm.txt")
normalize_file("24.1mb.cov.txt", "24.1mb.rpm.txt")

In [ ]:
def plot_sample(file, title):
    df = pd.read_csv(file, sep="\t")

    # среднее значение RPM по хромосоме
    chr_mean = df.groupby("chr")["rpm"].mean()

    plt.figure(figsize=(12,5))
    plt.bar(chr_mean.index, chr_mean.values)

    plt.xlabel("Chromosome")
    plt.ylabel("Normalized reads")
    plt.title(title)

    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()


plot_sample("19.1mb.rpm.txt", "Sample 19")
plot_sample("20.1mb.rpm.txt", "Sample 20")

In [ ]:
df19 = pd.read_csv("19.1mb.rpm.txt", sep="\t")
df20 = pd.read_csv("20.1mb.rpm.txt", sep="\t")

# объединяем по координатам бинов
df = pd.merge(
    df19[["chr","start","end","rpm"]],
    df20[["chr","start","end","rpm"]],
    on=["chr","start","end"],
    suffixes=("_19","_20")
)
df = df[(df["rpm_19"] > 0) | (df["rpm_20"] > 0)]
# лог трансформация
df["log19"] = np.log2(df["rpm_19"] + 1)
df["log20"] = np.log2(df["rpm_20"] + 1)

# коэффициент корреляции
r, p = pearsonr(df["log19"], df["log20"])

plt.figure(figsize=(6,6))

plt.scatter(
    df["log19"],
    df["log20"],
    s=5,
    alpha=0.4
)

plt.xlabel("Sample 19 log2(RPM+1)")
plt.ylabel("Sample 20 log2(RPM+1)")
plt.title(f"Correlation between samples (r = {r:.3f})")

plt.tight_layout()
plt.show()

In [ ]:
#z-score подсчет
df["z_19"] = zscore(df["rpm_19"])
df["z_20"] = zscore(df["rpm_20"])

In [ ]:
df["mid"] = (df["start"] + df["end"]) / 2

chromosomes = df["chr"].unique()

offset = 0
genome_pos = []
ticks = []
labels = []

for chrom in chromosomes:

    sub = df[df["chr"] == chrom]

    genome_pos.extend(sub["mid"] + offset)

    ticks.append(offset + sub["mid"].median())
    labels.append(chrom)

    offset += sub["end"].max()

df["genome_pos"] = genome_pos


plt.figure(figsize=(18,6))

colors = ["#4C72B0", "#DD8452"]

for i, chrom in enumerate(chromosomes):

    sub = df[df["chr"] == chrom]

    plt.scatter(
        sub["genome_pos"],
        sub["z_19"],      
        s=4,
        color=colors[i % 2],
        alpha=0.6
    )

plt.xticks(ticks, labels, rotation=90)

plt.xlabel("Chromosome position")
plt.ylabel("Z-score")
plt.title("Genome-wide Z-score (sample 19)")

plt.axhline(2, color="red", linestyle="--")
plt.axhline(-2, color="red", linestyle="--")

plt.tight_layout()
plt.show()